# 第13回: Model Context Protocol (MCP)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session13/session13_mcp.ipynb)

これまで扱ってきた Tool は、主にエージェントのアプリケーション内部に組み込まれたビルトインツール（内部ツール）だった。これに対して MCP は、アプリケーションの境界を越え、外部の機能やリソースをモデルが利用するための標準化されたプロトコルである。AI エージェントにとっては、MCP 経由で接続する能力をプラグインツール（外部ツール）と位置づけられる。

MCP Server が Tool・Resource・Prompt を公開し、MCP Client を持つエージェントアプリケーションが、それらを発見して利用する。共通の接続方式を用いることで、外部の能力を異なるエージェントやモデルから再利用しやすくなる。

---


## 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
# @title 必要なパッケージをインストールする
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session13

%pip install -q --break-system-packages "fastmcp>=4" "langchain[mcp]>=1.4" "langgraph>=1.2" "langchain-openai>=0.2"

In [ ]:
# @title APIキーの設定
import getpass
import os
from pathlib import Path

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')

---

## 1. MCP のアーキテクチャ

MCP は Client–Server アーキテクチャを取る。LLM と MCP Client を含むアプリケーション全体を **Host** と呼ぶ。


![MCPのアーキテクチャ](./assets/mcp-architecture.png)

| 構成要素 | 責務 | 例 |
|---|---|---|
| Host | LLM、権限、会話、UI、複数 Client を管理する | IDE、デスクトップアプリ、AI エージェント |
| MCP Client | 1つの Server と接続し、発見・呼び出しを仲介する | Host 内の MCP 接続コンポーネント |
| MCP Server | 外部能力を MCP の形式で公開する | Slack MCPサーバー、GitHub MCPサーバー、ローカルMCPサーバー |
| 外部サービス | 実際にデータ取得や操作を行う | Slack、GitHub、ローカルファイル |

### Transport と配置

| Transport | 配置 | 特徴 | 主な用途 |
|---|---|---|---|
| STDIO | Client と同じマシン | Client がServerを子プロセスを起動することが可能。標準入出力で JSON-RPC を交換 | ローカルファイル、開発ツール、個人環境 |
| Streamable HTTP | 別プロセス／別ホスト | HTTP 上の MCP endpoint へ接続し、複数 Client で共有できる | 組織サービス、SaaS、スケールが必要な環境 |

### MCPサーバーが提供するプリミティブ

MCP Server が公開する中心的なプリミティブは3つ。

| プリミティブ | 意味 | 主な選択者 | 例 |
|---|---|---|---|
| **Tool** | 副作用を含み得る実行可能な機能 | LLM | メール送信、DB検索、レコード更新 |
| **Resource** | URI で参照するデータや文脈 | アプリケーション | 文書、DBレコード、設定情報 |
| **Prompt** | 再利用できる対話テンプレート | ユーザー／アプリケーション | コードレビュー手順、週報作成テンプレート |

**実行**なら Tool、**参照する文脈**なら Resource、**定型的なやり取り**なら Prompt、という責務から選ぶ。

### MCP と Tool function calling

MCP と function calling は異なる層を担当する。

![MCP Tool vs Function Calling](./assets/mcp-tool-function-calling.png)

| 観点 | Tool function calling | MCP |
|---|---|---|
| 主な責務 | モデルが関数と引数を選ぶ | 能力を公開・発見・呼び出す |
| 境界 | LLM ↔ アプリケーション | MCP Client ↔ MCP Server |
| 標準化 | モデル提供者ごとに形式が異なる | オープンな共通プロトコル |
| Discovery | アプリが関数一覧をモデルへ渡す | Client が Server の能力一覧を取得する |
| 再利用 | アプリ内の統合になりやすい | Server を複数 Host から利用できる |

[![](https://mermaid.ink/img/pako:eNrtVltv2zYU_isC-5IA8kWSZRdsZsxLUbSAgxpJsAGb9kBJlESEFgWSbuw6_u_lRZYlW9uyAuvTbMCQz3cu3_l4SGoPEpZiAEFG2XNSIC6d5X1UOuojNnHOUVU4BRPyjwjcxPO725XzUf1zrhafnEVVUZIgSVh5fTOK5xH40wbqT0IJLqV3bvDPDYE14DK1D3VpipKngcD8C-bWkhNZbOKOibIE0Y7F_taJrxrGt8ZgKV63ffxLn5NLm4P2ExUqHSF3FP8UgZSIiqIdJCUlJR5kFG_fIUryckAkXguYqGSYv8tRBafVNgLzbnjGSjkQ5CuG_lsDZyiGGRqYmjcj7Tt3amYPhoFmXwMNw44mP4qiLfpqjvUc_O1itBfyB7RhhK4J_0MXhp5CrwzF6w7rziA5N4PBYP7y8fFx9dIzvfVIdrx6JroWq3YTMiXspWfM2xbr64iEk0peDm6darH69OIQLVSJj5uyU7_PT-3VOqHW0pwCTkYohW-yUH9dITl7wvBNEAT18-CZpLKAvlLaRhpaP-_JOodOBAopKwFHI1SRIUlYSbLdMMVCLeYoQTxmJbQBQ_Elj4DrFDpoOtaPyluVQIqbtrEyAgdbounqNVUEplkhJDQSfU8V_zVVYgKtuP-mQltqq4LT0XRcbV0rvwov1XhyNSrtoEaI74zzXxUHXJBzkgJl3WAXrDFfI_0X7HXOCMgCr3EEdHMpztCGStV1C_oVcYJiioX22detm136Aa0J3dnQO0z4jrmOUJX1hJIsAtr3UKfa7m71VdXJYWhry2w8do_GApO8MH5hy2ruuc9cbzZzfdmiahtIdZ_RmrB2pDhXN9MHxe5BHSHazZucgSuUpqTMDXaqUFEm77HZW-lDhRK8wlyfVJZJ4yYK9vweSbREMaYayxAVuB_-vJGCpPgXxPsdl4aOxszStKFHIim-QKS2tnvzx12sv7WNwHdo-9tR7U5KhTXAKWK32JL2emsbUqalOtAbd989gVT326t6g7a4hefYPTst67gFaikWdd0z6o2Ix4XowR5J8vSX0KXAVsbkaYnL3LYYniG9vV-sijc9R9u9W-jQaL39X-v_WmtzEEXlQR2F6iXhd8bWx9OQs01eAGj3JthUKZL4PUHqLfrkovYo5rdsU0oAw1lgcgC4B1sAvcnbYTgNJsHU8z3f1-AOQN-bDieT2dT3J14YeqE3Pbjgq6k6Hs68WehPZuNZ4M2C0FcROCWS8Tv7Um_e7Q_fALA6x_4?type=png)](https://mermaid.ai/live/edit#pako:eNrtVltv2zYU_isC-5IA8kWSZRdsZsxLUbSAgxpJsAGb9kBJlESEFgWSbuw6_u_lRZYlW9uyAuvTbMCQz3cu3_l4SGoPEpZiAEFG2XNSIC6d5X1UOuojNnHOUVU4BRPyjwjcxPO725XzUf1zrhafnEVVUZIgSVh5fTOK5xH40wbqT0IJLqV3bvDPDYE14DK1D3VpipKngcD8C-bWkhNZbOKOibIE0Y7F_taJrxrGt8ZgKV63ffxLn5NLm4P2ExUqHSF3FP8UgZSIiqIdJCUlJR5kFG_fIUryckAkXguYqGSYv8tRBafVNgLzbnjGSjkQ5CuG_lsDZyiGGRqYmjcj7Tt3amYPhoFmXwMNw44mP4qiLfpqjvUc_O1itBfyB7RhhK4J_0MXhp5CrwzF6w7rziA5N4PBYP7y8fFx9dIzvfVIdrx6JroWq3YTMiXspWfM2xbr64iEk0peDm6darH69OIQLVSJj5uyU7_PT-3VOqHW0pwCTkYohW-yUH9dITl7wvBNEAT18-CZpLKAvlLaRhpaP-_JOodOBAopKwFHI1SRIUlYSbLdMMVCLeYoQTxmJbQBQ_Elj4DrFDpoOtaPyluVQIqbtrEyAgdbounqNVUEplkhJDQSfU8V_zVVYgKtuP-mQltqq4LT0XRcbV0rvwov1XhyNSrtoEaI74zzXxUHXJBzkgJl3WAXrDFfI_0X7HXOCMgCr3EEdHMpztCGStV1C_oVcYJiioX22detm136Aa0J3dnQO0z4jrmOUJX1hJIsAtr3UKfa7m71VdXJYWhry2w8do_GApO8MH5hy2ruuc9cbzZzfdmiahtIdZ_RmrB2pDhXN9MHxe5BHSHazZucgSuUpqTMDXaqUFEm77HZW-lDhRK8wlyfVJZJ4yYK9vweSbREMaYayxAVuB_-vJGCpPgXxPsdl4aOxszStKFHIim-QKS2tnvzx12sv7WNwHdo-9tR7U5KhTXAKWK32JL2emsbUqalOtAbd989gVT326t6g7a4hefYPTst67gFaikWdd0z6o2Ix4XowR5J8vSX0KXAVsbkaYnL3LYYniG9vV-sijc9R9u9W-jQaL39X-v_WmtzEEXlQR2F6iXhd8bWx9OQs01eAGj3JthUKZL4PUHqLfrkovYo5rdsU0oAw1lgcgC4B1sAvcnbYTgNJsHU8z3f1-AOQN-bDieT2dT3J14YeqE3Pbjgq6k6Hs68WehPZuNZ4M2C0FcROCWS8Tv7Um_e7Q_fALA6x_4)

---


## 2. Discovery から実行まで

### Tool を使う場合

![MCP Protocol](./assets/mcp-sequence-diagram.png)

1. **Server Discovery（任意）**
   Client は、必要に応じて `server/discover` を呼び出し、Server が対応するプロトコルバージョンや Capabilities、Server 情報を確認する。（①, ②）
   このとき Client は `_meta` に `protocolVersion`、`clientCapabilities`、必要に応じて `clientInfo` を含める。
   `server/discover` は必須ではなく、Server の情報を事前確認する必要がなければ直接 `tools/list` を呼び出せる。

2. **Tool Discovery**
   Client は `tools/list` を呼び出し、Server が提供する Tool 一覧を取得する。（③, ④）
   Server は Tool の名前、説明、入力スキーマなどに加え、必要に応じて `ttlMs` や `cacheScope` といったキャッシュ情報を返す。
   Client はこの情報を一定期間キャッシュできるため、毎回 `tools/list` を実行する必要はない。

3. **ユーザー要求の受付**
   User が Host に要求を入力する。Host は LLM を使って要求内容を解釈し、利用可能な Tool が必要かどうかを判断する。（⑤）

4. **Tool の選択と引数生成**
   Host / LLM は、Discovery で取得した Tool 情報の中からユーザー要求に適した Tool を選択し、その Tool に渡す引数を生成する。

5. **Tool 実行の指示**
   Host は MCP Client に対して、選択した Tool の実行を指示する。（⑥）必要に応じて Host 側でユーザー権限や実行ポリシーを確認し、危険な操作ではユーザー承認を求めることもある。

6. **`tools/call` の送信**
   Client は Server に `tools/call` を送信する。（⑦）
   リクエストには Tool の `name` と `arguments` に加え、`_meta` として `protocolVersion` や Client Capabilities など、そのリクエストを処理するために必要な情報を含める。

7. **追加情報が必要な場合：MRTR**
   Tool を実行するためにユーザー入力やその他の追加情報が必要な場合、Server は処理を完了させず、`resultType = input_required` を返す。（⑧）
   この応答には、Client に取得してほしい情報を表す `inputRequests` と、後続リクエストとの関連付けに使う `requestState` が含まれる。

8. **追加情報の取得と再実行**
   Client は `inputRequests` を Host に渡し（⑨）、Host が必要に応じて User に追加情報を求める。（⑩）
   User から入力を受け取った後（⑪）、Host はその結果を Client に返す。（⑫）
   Client は取得した `inputResponses` と `requestState` を付けて、同じ `tools/call` を新しいリクエストとして再実行する。（⑬）

9. **Tool の実処理**
   必要な入力が揃うと、Server は Tool の処理を実行する。
   Tool の内部では、必要に応じて外部 API、データベース、SaaS、社内システムなどの External System を呼び出す。（⑭）

10. **結果返却**
    External System から結果を受け取った Server は（⑮）、`resultType = complete` と Tool の実行結果を Client に返す。（⑯）
    Client はその結果を Host に渡し（⑰）、Host / LLM がユーザーへの回答生成に利用する。

11. **回答または次の Tool 実行**
    LLM は Tool の結果を使って User に回答する。（⑱）
    追加の処理が必要であれば、別の Tool を選択して再び `tools/call` を実行することもできる。


### メソッド一覧

| 分類           | method                     | 用途                                 |
| ------------ | -------------------------- | ---------------------------------- |
| Discovery    | `server/discover`          | Serverの対応バージョン・Capabilities等を取得    |
| Tools        | `tools/list`               | Tool一覧を取得                          |
| Tools        | `tools/call`               | Toolを実行                            |
| Resources    | `resources/list`           | Resource一覧を取得                      |
| Resources    | `resources/templates/list` | Resource Template一覧を取得             |
| Resources    | `resources/read`           | Resourceの内容を取得                     |
| Prompts      | `prompts/list`             | Prompt一覧を取得                        |
| Prompts      | `prompts/get`              | Promptの内容を取得                       |
| Completion   | `completion/complete`      | Prompt/Resource Templateの引数補完候補を取得 |
| Subscription | `subscriptions/listen`     | Serverからの変更通知を受け取るストリームを開始         |

---

## 3. 主なユースケース

MCP は、LLM が現在の情報を取得し、現実のシステムへ作用する場面で使える。

- **データベース連携**: BigQuery や社内 DB を検索し、レポート作成やレコード更新を行う
- **外部 API**: 天気、株価、CRM、メールなどを共通の接続方法で利用する
- **情報抽出**: 文書 Resource と検索 Tool を組み合わせ、質問に必要な箇所だけを取り出す
- **生成メディア**: 画像・動画・音声・音楽生成サービスを複数ステップの制作フローへ組み込む
- **複雑なワークフロー**: 顧客情報取得 → 文案生成 → 画像生成 → 承認 → メール送信を連携する
- **IoT・ロボティクス**: センサー情報を読み、許可された機器操作を実行する
- **金融業務**: 市場データ、コンプライアンス、レポーティングを統合する
- **独自 Tool**: 社内関数やレガシーサービスを、複数のエージェントから再利用可能にする

作用範囲が大きいほど、自然言語で操作できる便利さと事故時の影響が同時に増える。金融取引、機器制御、外部送信では、MCP 対応だけを完了条件にせず、承認・権限・監査・停止手段まで含めて設計する。

---

## 4. FastMCP で Server を作る

FastMCP は Python で MCP Server を実装する高水準フレームワークである。関数の型ヒントから入力スキーマを、docstring から Tool の説明を生成する。

まずはネットワークも LLM も使わず、同一 Python プロセス内で Server をテストする。

次の Server は3種類のプリミティブを公開する。

- `calculate_total`: 注文金額を決定的に計算する Tool
- `policy://returns`: 返品ポリシーを読む Resource
- `support_reply`: サポート返信を作るための Prompt

In [ ]:
# @title Tool・Resource・Prompt を持つ MCP Server
from fastmcp import Client, FastMCP

mcp = FastMCP('Seminar Demo Server')


@mcp.tool
def calculate_total(unit_price: int, quantity: int, discount_percent: int = 0) -> dict:
    """注文の合計金額を計算する。

    Args:
        unit_price: 1個あたりの価格（0以上の整数）。
        quantity: 個数（1以上の整数）。
        discount_percent: 割引率（0から100までの整数）。
    """
    if unit_price < 0:
        raise ValueError('unit_price は0以上にしてください')
    if quantity < 1:
        raise ValueError('quantity は1以上にしてください')
    if not 0 <= discount_percent <= 100:
        raise ValueError('discount_percent は0から100にしてください')

    subtotal = unit_price * quantity
    discount = subtotal * discount_percent // 100
    return {
        'subtotal': subtotal,
        'discount': discount,
        'total': subtotal - discount,
        'currency': 'JPY',
    }


@mcp.resource('policy://returns')
def returns_policy() -> str:
    """デモ用の返品ポリシーを返す。"""
    return (
        '# 返品ポリシー\n\n'
        '未使用の商品は、到着から14日以内であれば返品できます。\n'
        '返品前にサポート窓口で受付番号を取得してください。'
    )


@mcp.prompt
def support_reply(customer_name: str, issue: str) -> str:
    """返品についてのサポート返信を作るための依頼文を返す。"""
    return (
        f'{customer_name} 様への返信を作成してください。\n'
        f'問い合わせ: {issue}\n'
        '返品ポリシーを確認し、利用できる手順だけを簡潔に案内してください。'
    )

print('Server を定義しました')

In [ ]:
# @title Discovery: Server が公開する能力を確認する

# 同一Pythonプロセス内のインメモリ接続（ネットワーク通信なし）
async with Client(mcp) as client:
    tools = await client.list_tools()
    resources = await client.list_resources()
    prompts = await client.list_prompts()

print('Tools:')
for tool in tools:
    print(f'- {tool.name}: {tool.description.splitlines()[0]}')
    print('  input schema:', tool.input_schema)

print('\nResources:')
for resource in resources:
    print(f'- {resource.uri}: {resource.name}')

print('\nPrompts:')
for prompt in prompts:
    print(f'- {prompt.name}: {prompt.description}')

In [ ]:
# @title Tool・Resource・Prompt を MCP Client から利用する
async with Client(mcp) as client:
    tool_result = await client.call_tool(
        'calculate_total',
        {'unit_price': 1200, 'quantity': 3, 'discount_percent': 10},
    )
    resource_result = await client.read_resource('policy://returns')
    prompt_result = await client.get_prompt(
        'support_reply',
        {'customer_name': '山田', 'issue': '昨日届いた商品を返品したい'},
    )

print('Tool result:', tool_result.data)
print('\nResource result:')
print(resource_result[0].text)
print('\nPrompt result:')
print(prompt_result.messages[0].content.text)

---

## 5. MCP Server を常駐プロセスとして起動する

Server の実装は `src/mcp-server/server.py`。公開するのはファイル操作の3つの Tool。

| Tool | 用途 |
|---|---|
| `list_files` | 公開ディレクトリのファイル一覧を返す |
| `read_file` | テキストファイルを読む |
| `write_file` | テキストファイルへ書き込む |

In [ ]:
# @title 公開ディレクトリを準備する
managed_dir = Path('workspace')
managed_dir.mkdir(parents=True, exist_ok=True)
(managed_dir / 'sample.txt').write_text(
    'MCP Server から読めるサンプルファイル。\n'
    'このディレクトリだけが Agent へ公開される。\n',
    encoding='utf-8',
)

print('公開するディレクトリ:', managed_dir.resolve())
for path in sorted(managed_dir.rglob('*')):
    print('-', path.relative_to(managed_dir))

### Server を常駐させる

ターミナルから Server を起動する。

```bash
cd session13
python3 src/mcp-server/server.py
```

`http://127.0.0.1:8000/mcp` が MCP endpoint になる。FastMCP の HTTP endpoint は既定で `/mcp`。

公開ディレクトリを変えるなら環境変数で渡す。

```bash
MCP_FILES_ROOT=/path/to/dir python3 src/mcp-server/server.py
```

Server は起動したまま、以降のセルを実行する。

### Server を停止してポートを解放する

起動したターミナルで `Ctrl+C` を押す。プロセスが終わればポートも解放される。

バックグラウンドへ回して `Ctrl+C` が使えないときは、プロセスを特定して終了させる。`address already in use` は、前回の Server が残っている状態で再起動したときに出る。

```bash
# プロセス名から PID を調べる
pgrep -af "mcp-server/server.py"

# ポート 8000 をリッスンしているプロセスから調べる
ss -ltnp | grep ':8000'

kill <PID>
```

PID を調べずに終了させるなら、他の Python プロセスを巻き込まないよう具体的なパターンを渡す。

```bash
pkill -f "mcp-server/server.py"
```

解放されたかを確認する。`kill` の直後は TIME_WAIT で数秒残ることがあるため、再起動が失敗したら少し待つ。

```bash
ss -ltnp | grep ':8000' || echo "port 8000 は空いている"
```

### LLM を使わずに疎通を確認する

Agent を組む前に、Transport と Tool の定義だけを確認する。この段階で API キーは不要。

In [ ]:
# @title HTTP 経由で Server の能力を確認する
async with Client('http://127.0.0.1:8000/mcp') as http_client:
    http_tools = await http_client.list_tools()
    listed = await http_client.call_tool('list_files', {})

print('Tools:')
for tool in http_tools:
    print(f'- {tool.name}: {tool.description.splitlines()[0]}')
    print('  input schema:', tool.input_schema)

print('\nlist_files の結果:', listed.data)

# 公開範囲の外を指すパスは Server 側で拒否される
async with Client('http://127.0.0.1:8000/mcp') as http_client:
    try:
        await http_client.call_tool('read_file', {'relative_path': '../server.py'})
    except Exception as error:
        print('\n範囲外アクセスは拒否される:', type(error).__name__)

---

## 6. LangChain Agent から MCP Server に接続する

ここからは LangChain の Agent を MCP Client として使う。`langchain.mcp` の `MCPAdapter` が MCP Server へ接続し、公開された Tool を LangChain の Tool へ変換する。変換後は `create_agent` へそのまま渡せる。

Agent 側から見ると、Tool が Python 関数か MCP Server 由来かの区別は無くなる。

実装上の注意が3つある。

- `MCPAdapter` は非同期コンテキストマネージャー。Tool の呼び出しには接続が必要なため、Agent の構築と実行を `async with` のブロックの中で行う
- MCP 由来の Tool は非同期のみ。Agent の実行には `invoke` ではなく `ainvoke` を使う
- `langchain.mcp` は beta。インポート時に `LangChainBetaWarning` が出る

以前は `langchain-mcp-adapters` パッケージが同じ役割を担っていたが、MCP SDK v2（仕様 2026-07-28 系）に対応せず、`langchain` 本体の `langchain.mcp` へ移された。

In [ ]:
# @title Agent が使う LLM を選ぶ
from langchain_openai import ChatOpenAI

model_id = 'gpt-5.4-nano' # @param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
llm = ChatOpenAI(model=model_id, temperature=0)
print('モデル:', model_id)

In [ ]:
# @title MCP Server の Tool を LangChain Tool へ変換する
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter

MCP_URL = 'http://127.0.0.1:8000/mcp'

# 必要な Tool だけを Agent に見せる
READONLY_TOOLS = {'list_files', 'read_file'}

async with MCPAdapter(MCP_URL) as adapter:
    mcp_tools = await adapter.list_tools()

    print('Server が公開する Tool:')
    for tool in mcp_tools:
        print(f'- {tool.name} ({type(tool).__name__}): {tool.description.splitlines()[0]}')

    readonly_tools = [tool for tool in mcp_tools if tool.name in READONLY_TOOLS]
    print('\n読み取り専用 Agent へ渡す Tool:', [tool.name for tool in readonly_tools])

In [ ]:
# @title 読み取り専用 Agent を実行する
# Tool の呼び出しには接続が必要なため、Agent の実行まで同じブロックの中で行う
async with MCPAdapter(MCP_URL) as adapter:
    tools = await adapter.list_tools()

    readonly_agent = create_agent(
        model=llm,
        tools=[tool for tool in tools if tool.name in READONLY_TOOLS],
        system_prompt=(
            'ユーザーのファイル閲覧を支援する。'
            '公開されたディレクトリの中だけを扱い、範囲外のパスは要求しない。'
            'ファイルの内容を答えるときは read_file の結果だけに基づく。'
        ),
    )

    read_result = await readonly_agent.ainvoke({
        'messages': [
            {'role': 'user', 'content': '使えるファイルを一覧して、sample.txt の内容を教えてください。'},
        ],
    })

print(read_result['messages'][-1].content)

In [ ]:
# @title 書き込みを含む Agent を実行する
# @markdown 書き込みを承認する場合は True にする
APPROVE = True # @param {type:"boolean"}

async with MCPAdapter(MCP_URL) as adapter:
    writable_agent = create_agent(
        model=llm,
        tools=await adapter.list_tools(),
        system_prompt=(
            'ユーザーのファイル管理を支援する。'
            '公開されたディレクトリの中だけを扱う。'
            '書き込みの前に、対象のパスと書き込む内容を要約して伝える。'
        ),
    )

    # 1ターン目: 指示どおり、Agent は書き込む前に内容を確認してくる
    confirmation = await writable_agent.ainvoke({
        'messages': [
            {'role': 'user', 'content': 'notes.md に「MCPの接続確認完了」と書いてください。'},
        ],
    })
    print('--- Agent の確認 ---')
    print(confirmation['messages'][-1].content)

    # 2ターン目: 承認または却下を返す。判断は Agent ではなく実行側が持つ
    answer = 'はい、その内容で作成してください。' if APPROVE else 'いいえ、書き込まないでください。'
    write_result = await writable_agent.ainvoke({
        'messages': [*confirmation['messages'], {'role': 'user', 'content': answer}],
    })

print(f'\n--- 実行側の判断: APPROVE={APPROVE} ---')
print(write_result['messages'][-1].content)